In [1]:
import matplotlib.pyplot as plt
import pickle

from pmbrl.model2 import Model
from pmbrl.data import Experiment_Data, get_data_expanded

In [2]:
nome_do_arquivo = 'triplet.pkl'

with open(nome_do_arquivo, 'rb') as arquivo:
    exp = pickle.load(arquivo)
    data = exp['data']
    model = exp['model']

del exp
del arquivo

In [3]:
expansions = {
    'estimated_p': ['p0', 'p1'],
    's_': ['s0', 's1', 's2', 's3'],
    's__': ['s_0', 's_1', 's_2', 's_3'],
}

df = get_data_expanded(data.evaluation_data, expansions)
df_avgs = df.groupby('episode')[['p0', 'p1']].mean().reset_index()

def get_avg_per_epi(row):
    return df_avgs.loc[df_avgs['episode'] == row.episode][['p0', 'p1']].values[0]
df[['avg_p0', 'avg_p1']] = df.apply(get_avg_per_epi, axis=1, result_type='expand')

df[['episode', 'avg_p0', 'avg_p1']]

,episode,avg_p0,avg_p1
0,0,0.539667,-4.429500
1,0,0.539667,-4.429500
2,0,0.539667,-4.429500
3,0,0.539667,-4.429500
4,0,0.539667,-4.429500
...,...,...,...
2122,99,0.208933,0.910067
2123,99,0.208933,0.910067
2124,99,0.208933,0.910067
2125,99,0.208933,0.910067


In [ ]:
import torch
import torch.nn as nn

m = model.transition_estimator.state_layer

input_values = torch.tensor(df[['s0', 's1', 's2', 's3', 'a_', 'avg_p0', 'avg_p1']].values) 
target_value = torch.tensor(df[['s_0', 's_1', 's_2', 's_3']].values)

for p in m.parameters():
    p.requires_grad = False

output = m(input_values.float()) 


def normilize(v): 
        mins = input_values[:,0:4].min(axis=0).values.repeat((v.shape[0], 1))
        maxs = input_values[:,0:4].max(axis=0).values.repeat((v.shape[0], 1))
        rang = maxs - mins
        return (v - mins) / rang
loss = nn.MSELoss(reduction='none')(normilize(output).float(), normilize(target_value).float())
# loss = nn.MSELoss(reduction='none')(normilize(output).float(), (target_value).float())
loss = torch.sqrt(loss.mean(axis=0).sum())
loss

tensor(0.0253)

In [5]:
results = data.get_evaluation_metrics()

results['rmse'] = results['rse_s0'] + results['rse_s1'] + results['rse_s2'] + results['rse_s3']
results.rmse.mean()

np.float64(0.18998166431593794)